# 🐼 Pandas + s3fs + Parquet

This notebook demonstrates the **Medallion Architecture** (Bronze → Silver → Gold) using Pandas, s3fs, and Parquet on top of RustFS S3 storage.

In [ ]:
import boto3


# Connect to local RustFS via boto3 (synchronous, no asyncio issues)
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1'
)

print("boto3 connected to RustFS!")

In [ ]:
from IPython.display import display
import pandas as pd
import os
import tempfile

# Create raw sales DataFrame
vendas_raw = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    'customer_id': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008', 'C009', 'C010'],
    'amount': [150.0, 200.0, 89.90, 320.0, 45.50, 670.0, 120.0, 55.0, 430.0, 999.99],
    'status': ['PAGO', 'PAGO', 'CANCELADO', 'PAGO', 'ESTORNADO', 'PAGO', 'CANCELADO', 'PAGO', 'PAGO', 'ESTORNADO'],
    'order_date': [
        '2025-01-10', '2025-01-11', '2025-01-12', '2025-01-13', '2025-01-14',
        '2025-01-15', '2025-01-16', '2025-01-17', '2025-01-18', '2025-01-19',
    ],
})

display(vendas_raw)

# Write raw CSV to Bronze layer via temp file
tmpdir = tempfile.mkdtemp()
local_csv = os.path.join(tmpdir, 'vendas_raw.csv')
vendas_raw.to_csv(local_csv, index=False)
s3.upload_file(local_csv, 'bronze', 'vendas_raw.csv')
print("Data ingested to Bronze!")

In [ ]:
from IPython.display import display

# Download CSV from Bronze via temp file
local_csv = os.path.join(tmpdir, 'vendas_raw.csv')
s3.download_file('bronze', 'vendas_raw.csv', local_csv)
vendas_raw = pd.read_csv(local_csv)

# Filter only paid orders (PAGO)
vendas_clean = vendas_raw[vendas_raw['status'] == 'PAGO'].copy()

# Add calculated column
vendas_clean['total_tax'] = vendas_clean['amount'] * 0.1

display(vendas_clean)

# Write cleaned data as Parquet to Silver layer
local_parquet = os.path.join(tmpdir, 'vendas_clean.parquet')
vendas_clean.to_parquet(local_parquet, index=False, engine='pyarrow')
s3.upload_file(local_parquet, 'silver', 'vendas_clean.parquet')
print("Data processed to Silver!")

In [ ]:
from IPython.display import display

# Download Parquet from Silver via temp file
local_parquet = os.path.join(tmpdir, 'vendas_clean.parquet')
s3.download_file('silver', 'vendas_clean.parquet', local_parquet)
vendas_clean = pd.read_parquet(local_parquet)

# Aggregation: total revenue and average ticket by status
vendas_agg = vendas_clean.groupby('status').agg(
    total_revenue=('amount', 'sum'),
    average_ticket=('amount', 'mean'),
).reset_index()

display(vendas_agg)

# Write aggregated data as Parquet to Gold layer
local_agg = os.path.join(tmpdir, 'vendas_agg.parquet')
vendas_agg.to_parquet(local_agg, index=False, engine='pyarrow')
s3.upload_file(local_agg, 'gold', 'vendas_agg.parquet')
print("Gold layer ready!")

In [ ]:
# List all objects across the three layers using boto3
for bucket in ['bronze', 'silver', 'gold']:
    resp = s3.list_objects_v2(Bucket=bucket)
    objs = resp.get('Contents', [])
    print(f"\n{s3://{bucket}}:")
    for obj in objs:
        print(f"  - {obj['Key']} ({obj['Size']} bytes)")

print("\nMedallion pipeline complete!")

In [ ]:
# Cleanup: delete all files created
s3.delete_object(Bucket='bronze', Key='vendas_raw.csv')
print("Deleted bronze/vendas_raw.csv")

s3.delete_object(Bucket='silver', Key='vendas_clean.parquet')
print("Deleted silver/vendas_clean.parquet")

s3.delete_object(Bucket='gold', Key='vendas_agg.parquet')
print("Deleted gold/vendas_agg.parquet")

import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print("Temp files cleaned.")
print("Cleanup done!")